In [2]:
!pip install "unstructured[pdf]"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 19.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.2/542.2 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 15.3 MB/s eta 0:00:00
   

In [180]:
from unstructured.partition.pdf import partition_pdf
import numpy as np
import pandas as pd
import spacy

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files


import joblib
import os

In [181]:
df = pd.read_csv("/content/Resume.csv", engine="python", on_bad_lines="skip")


df.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [109]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91 entries, 0 to 90
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ID           91 non-null     int64 
 1   Resume_str   91 non-null     object
 2   Resume_html  91 non-null     object
 3   Category     91 non-null     object
dtypes: int64(1), object(3)
memory usage: 3.0+ KB


# **Take CV (Extract text from PDF file)**

In [182]:
pdf_path = "/content/MOSTAFA_NABIL_CV.pdf"
elements = partition_pdf(filename=pdf_path, strategy="auto")
resume_text = "\n\n".join([str(e) for e in elements])

In [19]:
resume_text

'Mostafa Nabil Ali Data Scientist\n\nmostafan22034405 @gmail.com\n\n+201110076652\n\nCairo, Egypt\n\nlinkedin.com/in/mostafa-nabil-95844130a\n\nSUMMARY\n\nEntry-level Data Scientist with a solid Computer Science background and hands-on experience in machine learning, computer vision, and AI- powered applications. Skilled in Python, SQL, predictive modeling, and model evaluation. Delivered impactful projects including an AI fitness assistant achieving R² 0.97 and a fraud detection system with AUC-ROC 0.986. Eager to apply end-to-end data science skills to deliver actionable insights and drive business value.\n\nEDUCATION\n\nBachelor in Computer Science Arab Academy for Science, Technology & Maritime Transport\n\n10/2021 – 07/2025 Egypt\n\nMajor: Computer Science GPA : 3.4\n\nEXPERIENCE\n\nAI Trainee Universitat Autònoma de Barcelona, Barcelona, Spain | July 2024 – August 2024\n\nApplied computer vision and data analysis techniques to AI projects, working on data preprocessing, model eva

# **Take Job Description (Raw string text)**

In [183]:
job_description_text = """
Data Scientist (Entry-Level)

Location: Cairo, Egypt
Employment Type: Full-time

About the Role
We are looking for an Entry-Level Data Scientist with a strong computer science background and hands-on experience in machine learning and computer vision. The ideal candidate will be skilled in data analysis, model development, and delivering actionable insights to support business decisions.

Key Responsibilities
- Perform data preprocessing, cleaning, and exploratory data analysis (EDA) to uncover trends and patterns.
- Build and evaluate predictive and classification models using machine learning algorithms such as Random Forest, SVM, Decision Trees, and KNN.
- Apply computer vision and natural language processing techniques including pose estimation, tokenization, and lemmatization.
- Create clear and impactful visualizations and reports using Matplotlib and Seaborn.
- Collaborate with teams to integrate AI-driven solutions into applications and workflows.

Qualifications
- Bachelor’s degree in Computer Science or related field.
- Strong programming skills in Python and SQL.
- Knowledge of feature engineering, model evaluation, and statistical analysis.

Soft Skills
- Strong analytical and problem-solving abilities.
- Ability to communicate technical insights clearly to non-technical stakeholders.
"""

# **Clean & Preprocess Both Texts and dataset**

In [184]:
nlp = spacy.load("en_core_web_sm")

def preprocess_Text(text):
  doc = nlp(text.lower())
  clean_tokens = [
        token.lemma_ for token in doc
        if not token.is_stop and not token.is_punct and token.is_alpha
    ]
  return " ".join(clean_tokens)

In [185]:
cleaned_resume = preprocess_Text(resume_text)
cleaned_jd = preprocess_Text(job_description_text)

In [187]:
resume_texts = df['Resume_str'].tolist()
cleaned_corpus = [preprocess_Text(text) for text in resume_texts]

# **Create and fit TF-IDF vectorizer pipeline**

In [188]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 8), max_features=6000, min_df=2))
])

pipeline.fit(cleaned_corpus)

tfidf_matrix = pipeline.transform([cleaned_resume, cleaned_jd])


# **Calculating the similarity**

In [189]:
score = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
print(f"Sample Cosine Similarity Score: {score * 100:.2f}%")

Sample Cosine Similarity Score: 54.71%


**Downloads the pre-fitted Scikit-Learn TF-IDF pipeline so it can be integrated into the Flask web application.**

In [190]:
joblib.dump(pipeline, 'pipeline.joblib')

files.download('pipeline.joblib')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Load a pre-trained Sentence-BERT model

In [191]:
model = SentenceTransformer('all-MiniLM-L6-v2') # from huggingface

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generate embeddings for the cleaned resume and job description

In [192]:
resume_embedding = model.encode(cleaned_resume)
jd_embedding = model.encode(cleaned_jd)

Reshape for cosine similarity calculation (sklearn expects 2D arrays)

In [193]:
resume_embedding_2d = resume_embedding.reshape(1, -1)
jd_embedding_2d = jd_embedding.reshape(1, -1)

# **Calculate cosine similarity**

In [194]:
semantic_score = cosine_similarity(resume_embedding_2d, jd_embedding_2d)[0][0]
print(f"Sentence-BERT Semantic Similarity Score: {semantic_score * 100:.2f}%")

Sentence-BERT Semantic Similarity Score: 81.34%


In [195]:
joblib.dump(model, 'BERT_model.joblib')

files.download('BERT_model.joblib')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# **Final Score**

In [215]:
tfidf_score = round(min((score / 0.45)* 100, 100.00), 2)

final_score = round((0.65 * (semantic_score * 100)) + (0.35 * tfidf_score), 2)

print(f"Final Score: {final_score}%")

Final Score: 87.87000274658203%




*   tfidf as skill matches score

*   Context & Experience Match for bert

*   Candidate Match Score for final score




